# Machine Learning and MSTR Market Regimes

A machine-learning directional strategy configured for MSTR with walk-forward validation.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
========================================================================
PRODUCTION-READY ML REGIME-ADAPTIVE TRADING STRATEGY
WITH FEATURE SELECTION AND MULTI-TICKER VALIDATION
========================================================================

ADDITIONS:
1. FeatureSelector class - reduces overfitting by selecting top N features
2. MultiTickerValidator class - validates strategy across multiple assets
3. Updated XGBoost params - reduced complexity (max_depth=3, min_child_weight=3)
4. Early stopping in model training
5. Feature selection integrated into walk-forward validation

ALL ORIGINAL FEATURES PRESERVED - Feature selection happens AFTER calculation
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from scipy.stats import t, norm
from scipy.optimize import minimize
from hmmlearn import hmm
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")


class StrategyConfig:
    """
    Centralized configuration to eliminate hardcoded parameters.
    All strategy parameters are defined here for easy modification.

    UPDATED: Reduced XGBoost complexity to prevent overfitting
    """

    BACKTEST_START = "2013-01-01"
    BACKTEST_END = "2024-12-31"
    WARMUP_MONTHS = 24
    SYMBOL_MAIN = "MSTR"
    SYMBOLS_CORR = ["BTCUSD", "SPY"]
    BENCHMARK = "URTH"

    @property
    def START_DATE(self):
        start = pd.to_datetime(self.BACKTEST_START)
        return (start - pd.DateOffset(months=self.WARMUP_MONTHS)).strftime("%Y-%m-%d")

    @property
    def END_DATE(self):
        return self.BACKTEST_END

    RETURN_WINDOWS = [5, 10, 20, 60]
    YZRV_WINDOW = 20
    HAR_WINDOWS = {"daily": 1, "weekly": 5, "monthly": 22}
    CORR_WINDOWS = [20, 60]
    MOMENTUM_WINDOWS = [20, 60, 120]
    TSMOM_WINDOWS = [20, 60, 120]
    ENTROPY_WINDOW = 20
    ENTROPY_BINS = 10
    HURST_WINDOWS = [60, 120]
    AMIHUD_WINDOW = 20
    N_REGIMES = 3
    HMM_COVARIANCE = "full"
    HMM_N_ITER = 100
    HMM_RANDOM_STATE = 42
    XGB_PARAMS = {
        "n_estimators": 200,
        "max_depth": 3,
        "learning_rate": 0.03,
        "subsample": 0.7,
        "colsample_bytree": 0.7,
        "min_child_weight": 3,
        "gamma": 0.1,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "random_state": 42,
    }
    VOL_NORMALIZATION = True
    VOL_LOOKBACK = 20
    REFIT_FREQUENCY = 20
    PREDICTION_HORIZON = 5
    MIN_TRAIN_DAYS = 252
    USE_FEATURE_SELECTION = True
    N_FEATURES_SELECT = 15
    INITIAL_CAPITAL = 100000.0
    COMPOUNDING_MODE = False
    LEVERAGE = 1.0
    COMMISSION_PCT = 0.0005
    SLIPPAGE_BPS = 5
    MAX_POSITION_SIZE = 1.0
    USE_CVAR_STOP = True
    CVAR_LOOKBACK = 60
    CVAR_QUANTILE = 0.95
    CVAR_MULTIPLIER = 1.5
    N_TESTS_ADJUSTMENT = 100
    CONFIDENCE_LEVEL = 0.95
    CDAR_QUANTILE = 0.95
    PLOT_FIGSIZE = (15, 10)
    PLOT_DPI = 100


class DataFetcher:
    """
    Handles all data fetching from Yahoo Finance.
    Uses adjusted close prices to handle splits and dividends.
    """

    def fetch_ohlcv(self, symbol, start_date, end_date):
        """
        Fetch OHLCV data for a given symbol.

        Parameters:
        -----------
        symbol : str
            Ticker symbol (e.g., 'MSTR', 'SPY')
        start_date : str
            Start date in 'YYYY-MM-DD' format
        end_date : str
            End date in 'YYYY-MM-DD' format

        Returns:
        --------
        pd.DataFrame
            DataFrame with OHLCV data, indexed by date
        """
        try:
            response_prices = {"historical": download_price_records(symbol, start_date, end_date)}
            data = response_prices
            if "historical" not in data:
                raise ValueError(f"No data returned for {symbol}")
            df = pd.DataFrame(data["historical"])
            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values("date").set_index("date")
            if "adjClose" in df.columns:
                df["close"] = df["adjClose"]
            cols = ["open", "high", "low", "close", "volume"]
            df = df[[c for c in cols if c in df.columns]]
            print(f"✓ Fetched {len(df)} rows for {symbol}")
            return df
        except Exception as e:
            print(f"✗ Error fetching {symbol}: {str(e)}")
            return pd.DataFrame()


class FeatureEngine:
    """
    Comprehensive feature engineering with strict temporal controls.
    All features use explicit shift() operations to prevent look-ahead bias.

    ALL ORIGINAL FEATURES PRESERVED - No simplification
    """

    def __init__(self, config):
        self.config = config

    def compute_basic_returns(self, df):
        """
        Compute basic return features with proper temporal alignment.

        Returns computed:
        - log_ret: Log return from Close[T-1] to Close[T]
        - ooc_ret: Open-to-Close return (intraday)
        - ooo_ret: Open[T-1] to Open[T] (overnight gap)
        """
        df["log_ret"] = np.log(df["close"] / df["close"].shift(1))
        df["ooc_ret"] = np.log(df["close"] / df["open"])
        df["ooo_ret"] = np.log(df["open"] / df["open"].shift(1))
        return df

    def compute_rolling_returns(self, df):
        """
        Compute rolling sum of returns over various windows.
        These capture momentum at different timescales.
        """
        for w in self.config.RETURN_WINDOWS:
            df[f"log_ret_roll_{w}"] = df["log_ret"].rolling(w).sum()
            df[f"ooc_roll_{w}"] = df["ooc_ret"].rolling(w).sum()
            df[f"ooo_roll_{w}"] = df["ooo_ret"].rolling(w).sum()
        return df

    def compute_yang_zhang_volatility(self, df):
        """
        Yang-Zhang Realized Volatility Estimator.

        This is 14x more efficient than close-to-close volatility.
        It combines overnight, intraday, and high-low range information.

        Formula:
        σ²_YZ = σ²_o + k·σ²_c + (1-k)·σ²_rs

        where:
        - σ²_o: Overnight variance (open/close[-1])
        - σ²_c: Close-to-close variance
        - σ²_rs: Rogers-Satchell variance (using high/low)
        - k: Scaling factor based on window size
        """
        window = self.config.YZRV_WINDOW
        ro = np.log(df["open"] / df["close"].shift(1))
        rc = np.log(df["close"] / df["open"])
        rs = np.log(df["high"] / df["low"])
        sigma_o = ro.rolling(window).var()
        sigma_c = rc.rolling(window).var()
        sigma_rs = (rs**2).rolling(window).mean()
        k = 0.34 / (1.34 + (window + 1) / (window - 1))
        df["yzrv"] = np.sqrt(sigma_o + k * sigma_c + (1 - k) * sigma_rs)
        return df

    def compute_har_structure(self, df):
        """
        Heterogeneous Autoregressive (HAR) structure on YZRV.

        HAR model decomposes volatility into daily, weekly, and monthly components.
        This captures multi-scale volatility dynamics.
        """
        df["yzrv_d"] = df["yzrv"]
        df["yzrv_w"] = df["yzrv"].rolling(self.config.HAR_WINDOWS["weekly"]).mean()
        df["yzrv_m"] = df["yzrv"].rolling(self.config.HAR_WINDOWS["monthly"]).mean()
        df["har_signal"] = df["yzrv_d"] + df["yzrv_w"] + df["yzrv_m"]
        return df

    def compute_cross_correlations(self, df, corr_assets):
        """
        Compute rolling correlations with other assets (BTC, SPY).

        Cross-asset correlations help identify regime shifts and
        diversification opportunities.
        """
        main_ret = df["log_ret"]
        for asset_name, asset_df in corr_assets.items():
            asset_ret = np.log(asset_df["close"] / asset_df["close"].shift(1))
            df[f"{asset_name}_ret"] = asset_ret
            for w in self.config.CORR_WINDOWS:
                df[f"corr_{asset_name}_{w}"] = main_ret.rolling(w).corr(df[f"{asset_name}_ret"])
        return df

    def compute_amihud_illiquidity(self, df):
        """
        Amihud Illiquidity Measure.

        Measures price impact per dollar traded.
        Higher values indicate less liquid markets.

        Formula: |Return| / Volume (averaged over window)
        """
        df["amihud"] = (
            (df["log_ret"].abs() / df["volume"]).rolling(self.config.AMIHUD_WINDOW).mean()
        )
        return df

    def compute_momentum(self, df):
        """
        Simple momentum: (Price[T] / Price[T-W]) - 1

        Captures trend strength over various lookback periods.
        """
        for w in self.config.MOMENTUM_WINDOWS:
            df[f"momentum_{w}"] = df["close"] / df["close"].shift(w) - 1
        return df

    def compute_tsmom(self, df):
        """
        Time Series Momentum (TSMOM).

        Sign of average return over lookback window.
        Simple but effective trend-following signal.
        """
        for w in self.config.TSMOM_WINDOWS:
            df[f"tsmom_{w}"] = np.sign(df["log_ret"].rolling(w).mean())
        return df

    def compute_entropy(self, df):
        """
        Shannon Entropy of returns distribution.

        Measures unpredictability/randomness in returns.
        High entropy = more random/uncertain market.
        Low entropy = more predictable/structured market.

        Formula: H = -Σ p(x) * log(p(x))
        """
        window = self.config.ENTROPY_WINDOW
        n_bins = self.config.ENTROPY_BINS

        def rolling_entropy(x):
            """Calculate entropy for a rolling window of returns."""
            if len(x) < window or x.isna().all():
                return np.nan
            x_clean = x.dropna()
            if len(x_clean) < 5:
                return np.nan
            counts, _ = np.histogram(x_clean, bins=n_bins)
            probs = counts / counts.sum()
            probs = probs[probs > 0]
            entropy = -np.sum(probs * np.log(probs))
            return entropy

        df["entropy"] = df["log_ret"].rolling(window).apply(rolling_entropy, raw=False)
        return df

    def compute_hurst_exponent(self, df):
        """
        Rolling Hurst Exponent using R/S analysis.

        Hurst exponent measures long-term memory in time series:
        - H < 0.5: Mean-reverting series
        - H = 0.5: Random walk (no memory)
        - H > 0.5: Trending series (persistent)

        This helps identify market regimes:
        - Mean reversion strategies work better when H < 0.5
        - Momentum strategies work better when H > 0.5
        """

        def hurst_rs(x):
            """
            Calculate Hurst exponent using R/S (Rescaled Range) analysis.
            """
            if len(x) < 20 or x.isna().any():
                return np.nan
            y = x - x.mean()
            z = np.cumsum(y)
            R = z.max() - z.min()
            S = x.std()
            if S == 0 or R == 0:
                return np.nan
            rs = R / S
            n = len(x)
            H = np.log(rs) / np.log(n)
            return H

        for w in self.config.HURST_WINDOWS:
            df[f"hurst_{w}"] = df["log_ret"].rolling(w).apply(hurst_rs, raw=False)
        return df

    def compute_all_features(self, df, corr_assets):
        """
        Master function to compute all features in correct order.

        ALL ORIGINAL FEATURES COMPUTED - Nothing removed
        """
        print("\n" + "=" * 60)
        print("FEATURE ENGINEERING")
        print("=" * 60)
        print("Computing basic returns...")
        df = self.compute_basic_returns(df)
        print("Computing rolling volatility...")
        df["rolling_vol"] = df["log_ret"].rolling(self.config.VOL_LOOKBACK).std()
        print("Computing rolling returns...")
        df = self.compute_rolling_returns(df)
        print("Computing Yang-Zhang volatility...")
        df = self.compute_yang_zhang_volatility(df)
        print("Computing HAR structure...")
        df = self.compute_har_structure(df)
        print("Computing cross-asset correlations...")
        df = self.compute_cross_correlations(df, corr_assets)
        print("Computing Amihud illiquidity...")
        df = self.compute_amihud_illiquidity(df)
        print("Computing momentum features...")
        df = self.compute_momentum(df)
        print("Computing TSMOM signals...")
        df = self.compute_tsmom(df)
        print("Computing entropy features...")
        df = self.compute_entropy(df)
        print("Computing Hurst exponent...")
        df = self.compute_hurst_exponent(df)
        print(f"✓ Feature engineering complete. {len(df.columns)} features generated.")
        return df


class FeatureSelector:
    """
    Dynamic feature selection to reduce overfitting.

    CRITICAL: Selects features on FIRST fold only using training data.
    All original features are still computed - this only FILTERS them.

    FIXED: Now handles dynamically added features (like 'regime') correctly

    Strategy:
    1. Train simple Random Forest on first training fold
    2. Select top N features by importance
    3. Use same features across all folds (no look-ahead)
    4. Reduces dimensionality from ~30 to 15 features
    """

    def __init__(self, config):
        self.config = config
        self.selected_features = None
        self.feature_importance_df = None

    def select_features(self, X_train, y_train, feature_names, top_n=15):
        """
        Select most important features using Random Forest.

        FIXED: Now accepts X_train as DataFrame to get actual column names

        Uses a simple, regularized model to avoid selecting spurious features.

        Parameters:
        -----------
        X_train : pd.DataFrame or np.ndarray
            Training feature matrix
        y_train : np.ndarray
            Training labels
        feature_names : list
            Names of features (used only if X_train is numpy array)
        top_n : int
            Number of features to select

        Returns:
        --------
        list : Selected feature names
        """
        from sklearn.ensemble import RandomForestClassifier

        if isinstance(X_train, pd.DataFrame):
            actual_feature_names = list(X_train.columns)
            X_train_array = X_train.values
        else:
            actual_feature_names = feature_names
            X_train_array = X_train
        unique, counts = np.unique(y_train, return_counts=True)
        if len(unique) < 2:
            print("Warning: Only one class in training data. Using all features.")
            return actual_feature_names
        if X_train_array.shape[1] != len(actual_feature_names):
            print(f"ERROR: Feature count mismatch!")
            print(f"  X_train columns: {X_train_array.shape[1]}")
            print(f"  Feature names: {len(actual_feature_names)}")
            print("  Using all features without selection.")
            return actual_feature_names
        rf = RandomForestClassifier(
            n_estimators=100, max_depth=3, min_samples_leaf=20, max_features="sqrt", random_state=42
        )
        try:
            rf.fit(X_train_array, y_train)
        except Exception as e:
            print(f"Warning: Feature selection failed: {e}. Using all features.")
            return actual_feature_names
        self.feature_importance_df = pd.DataFrame(
            {"feature": actual_feature_names, "importance": rf.feature_importances_}
        ).sort_values("importance", ascending=False)
        self.selected_features = self.feature_importance_df.head(top_n)["feature"].tolist()
        print(f"\n{'=' * 60}")
        print("FEATURE SELECTION RESULTS")
        print("=" * 60)
        print(f"Total features available: {len(actual_feature_names)}")
        print(f"Features selected: {len(self.selected_features)}")
        print(f"Dimensionality reduction: {(1 - top_n / len(actual_feature_names)) * 100:.1f}%")
        print(f"\nTop 10 features by importance:")
        for i, row in self.feature_importance_df.head(10).iterrows():
            print(f"  {row['feature']:30s} {row['importance']:.4f}")
        categories = {
            "volatility": ["yzrv", "har", "rolling_vol"],
            "momentum": ["momentum", "tsmom", "log_ret_roll"],
            "correlation": ["corr_"],
            "microstructure": ["entropy", "hurst", "amihud"],
            "regime": ["regime"],
        }
        print(f"\nSelected features by category:")
        for cat_name, cat_keywords in categories.items():
            cat_features = [
                f for f in self.selected_features if any((kw in f for kw in cat_keywords))
            ]
            if cat_features:
                print(f"  {cat_name.capitalize()}: {len(cat_features)}")
        return self.selected_features

    def transform(self, X, feature_names=None):
        """
        Filter feature matrix to selected features only.

        FIXED: Now handles both DataFrame and numpy array inputs correctly

        Parameters:
        -----------
        X : pd.DataFrame or np.ndarray
            Feature matrix
        feature_names : list, optional
            Names of features in X (required if X is numpy array)

        Returns:
        --------
        Filtered feature matrix (same type as input)
        """
        if self.selected_features is None:
            raise ValueError("Must call select_features() first")
        if isinstance(X, pd.DataFrame):
            return X[self.selected_features]
        else:
            if feature_names is None:
                raise ValueError("feature_names required when X is numpy array")
            indices = [feature_names.index(f) for f in self.selected_features if f in feature_names]
            return X[:, indices]

    def get_feature_report(self):
        """Return feature importance report as DataFrame."""
        return self.feature_importance_df


class RegimeDetector:
    """
    Hidden Markov Model for regime detection.

    Uses YZRV and cross-asset correlations to identify market regimes:
    - Regime 0: Low volatility (calm markets)
    - Regime 1: Medium volatility (normal markets)
    - Regime 2: High volatility (stressed markets)
    """

    def __init__(self, config):
        self.config = config
        self.model = None
        self.scaler_mean = None
        self.scaler_std = None

    def prepare_regime_features(self, df):
        """
        Select and prepare features for HMM regime detection.

        Features used:
        - YZRV components (daily, weekly, monthly)
        - Cross-asset correlations (captures market integration)
        """
        regime_features = []
        if "yzrv_d" in df.columns:
            regime_features.append("yzrv_d")
        if "yzrv_w" in df.columns:
            regime_features.append("yzrv_w")
        if "yzrv_m" in df.columns:
            regime_features.append("yzrv_m")
        corr_cols = [c for c in df.columns if c.startswith("corr_") and "_20" in c]
        regime_features.extend(corr_cols)
        return regime_features

    def fit(self, df, regime_features):
        """
        Fit HMM model to training data.

        CRITICAL: Must use expanding window to prevent look-ahead bias.
        This function should only be called on training data.
        """
        X = df[regime_features].values
        valid_idx = ~np.isnan(X).any(axis=1)
        X_clean = X[valid_idx]
        if len(X_clean) < 100:
            raise ValueError("Insufficient clean data for HMM training")
        self.scaler_mean = X_clean.mean(axis=0)
        self.scaler_std = X_clean.std(axis=0)
        X_scaled = (X_clean - self.scaler_mean) / (self.scaler_std + 1e-08)
        self.model = hmm.GaussianHMM(
            n_components=self.config.N_REGIMES,
            covariance_type=self.config.HMM_COVARIANCE,
            n_iter=self.config.HMM_N_ITER,
            random_state=self.config.HMM_RANDOM_STATE,
        )
        self.model.fit(X_scaled)
        print(f"✓ HMM trained with {len(X_clean)} observations")
        return self

    def predict(self, df, regime_features):
        """
        Predict regimes for new data.

        Uses previously fitted model and scaler.
        """
        if self.model is None:
            raise ValueError("Model not fitted. Call fit() first.")
        X = df[regime_features].values
        valid_idx = ~np.isnan(X).any(axis=1)
        regimes = np.full(len(X), np.nan)
        if valid_idx.sum() == 0:
            return regimes
        X_valid = X[valid_idx]
        X_scaled = (X_valid - self.scaler_mean) / (self.scaler_std + 1e-08)
        regimes_valid = self.model.predict(X_scaled)
        regimes[valid_idx] = regimes_valid
        return regimes

    def fit_predict(self, df):
        """
        Convenience method: fit and predict in one step.
        Used during walk-forward validation.
        """
        regime_features = self.prepare_regime_features(df)
        self.fit(df, regime_features)
        regimes = self.predict(df, regime_features)
        return (regimes, regime_features)


class MLTradingModel:
    """
    XGBoost classifier with walk-forward expanding window validation.

    NEW: Integrated with FeatureSelector to reduce overfitting
    NEW: Early stopping in XGBoost training

    Temporal flow:
    - Train on [0, T]
    - Validate on [T+1, T+V]
    - Test on [T+V+1, T+V+T2]

    Model is retrained at each fold using expanding window.
    """

    def __init__(self, config):
        self.config = config
        self.models = []
        self.feature_names = None
        self.feature_selector = FeatureSelector(config)

    def prepare_ml_features(self, df):
        """
        Select features for ML model.

        Excludes:
        - Target variable
        - Price columns (open, high, low, close)
        - Volume (to prevent data leakage)
        - Intermediate return columns
        """
        exclude_cols = [
            "target",
            "target_direction",
            "target_magnitude",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "log_ret",
            "ooc_ret",
            "ooo_ret",
            "btcusd_ret",
            "spy_ret",
        ]
        feature_cols = [c for c in df.columns if c not in exclude_cols]
        return feature_cols

    def create_target(self, df, horizon=5):
        """
        Create volatility-adjusted target variable.

        Target: Return from Open[T] to Close[T+horizon], normalized by volatility
        """
        df["target_return_raw"] = df["close"].shift(-horizon) / df["open"] - 1
        if hasattr(self.config, "VOL_NORMALIZATION") and self.config.VOL_NORMALIZATION:
            df["target_return"] = df["target_return_raw"] / (df["rolling_vol"] + 1e-08)
        else:
            df["target_return"] = df["target_return_raw"]
        df["target"] = (df["target_return"] > 0).astype(int)
        df["target_magnitude"] = df["target_return"].abs()
        return df

    def walk_forward_validation(self, df):
        """
        Rolling walk-forward validation with feature selection.

        NEW: Feature selection on first VALID fold (prevents data snooping)
        NEW: Early stopping in XGBoost training (FIXED)

        CRITICAL: HMM regime detection integrated into walk-forward loop
        to prevent look-ahead bias. Regimes detected using only past data.
        """
        print("\n" + "=" * 60)
        print("ROLLING WALK-FORWARD VALIDATION + FEATURE SELECTION")
        print("=" * 60)
        feature_cols = self.prepare_ml_features(df)
        self.feature_names = feature_cols
        print("\nApplying temporal shift to features (T -> T-1)...")
        df[feature_cols] = df[feature_cols].shift(1)
        print("✓ Features shifted by 1 day to ensure no look-ahead bias")
        median_vol = df["rolling_vol"].median()
        print(f"\nAsset volatility detection:")
        print(f"  Median annualized vol: {median_vol * np.sqrt(252) * 100:.1f}%")
        if median_vol * np.sqrt(252) < 0.15:
            adjusted_horizon = int(self.config.PREDICTION_HORIZON * 2)
            adjusted_refit = int(self.config.REFIT_FREQUENCY * 2)
            print(f"  → Low volatility regime detected")
            print(
                f"  → Adjusting PREDICTION_HORIZON: {self.config.PREDICTION_HORIZON} → {adjusted_horizon}"
            )
            print(
                f"  → Adjusting REFIT_FREQUENCY: {self.config.REFIT_FREQUENCY} → {adjusted_refit}"
            )
        else:
            adjusted_horizon = self.config.PREDICTION_HORIZON
            adjusted_refit = self.config.REFIT_FREQUENCY
            print(f"  → Normal/high volatility regime")
            print(f"  → Using base parameters")
        if self.config.VOL_NORMALIZATION:
            print("\nApplying volatility normalization...")
            return_features = [
                c
                for c in feature_cols
                if any((x in c for x in ["ret", "momentum", "tsmom", "ooc", "ooo"]))
            ]
            for col in return_features:
                df[col] = df[col] / (df["rolling_vol"] + 1e-08)
            print(f"✓ Normalized {len(return_features)} return-based features by volatility")
            print(f"✓ StandardScaler will be applied per-fold for final standardization")
        regime_detector = RegimeDetector(self.config)
        regime_features = regime_detector.prepare_regime_features(df)
        df["regime"] = np.nan
        df = self.create_target(df, adjusted_horizon)
        valid_data = df[feature_cols + ["target", "rolling_vol"]].dropna().copy()
        print(f"Valid data points: {len(valid_data)}")
        print(f"Features: {len(feature_cols)}")
        print(f"Date range: {valid_data.index.min()} to {valid_data.index.max()}")
        print(f"Refit frequency: Every {adjusted_refit} days")
        print(f"Prediction horizon: {adjusted_horizon} days ahead")
        fold_results = []
        all_predictions = pd.Series(index=df.index, dtype=float)
        all_probabilities = pd.Series(index=df.index, dtype=float)
        train_start_idx = 0
        train_end_idx = self.config.MIN_TRAIN_DAYS
        fold_counter = 0
        selected_features = None
        feature_selection_done = False
        while train_end_idx < len(valid_data):
            test_start_idx = train_end_idx
            test_end_idx = min(test_start_idx + adjusted_horizon, len(valid_data))
            if test_start_idx >= test_end_idx:
                break
            fold_counter += 1
            X_train_df = valid_data.iloc[train_start_idx:train_end_idx][feature_cols].copy()
            y_train = valid_data.iloc[train_start_idx:train_end_idx]["target"].values
            X_test_df = valid_data.iloc[test_start_idx:test_end_idx][feature_cols].copy()
            y_test = valid_data.iloc[test_start_idx:test_end_idx]["target"].values
            unique_classes = np.unique(y_train)
            if len(unique_classes) < 2:
                print(
                    f"Fold {fold_counter}: Skipping - only {len(unique_classes)} class(es) in training"
                )
                train_end_idx += adjusted_refit
                continue
            unique, counts = np.unique(y_train, return_counts=True)
            minority_pct = counts.min() / len(y_train)
            if minority_pct < 0.02:
                print(f"Fold {fold_counter}: Skipping - extreme imbalance ({minority_pct:.1%})")
                train_end_idx += adjusted_refit
                continue
            if minority_pct < 0.3:
                class_weights = len(y_train) / (len(unique) * counts)
                sample_weights = np.array([class_weights[int(y)] for y in y_train])
                use_weights = True
                if fold_counter % 10 == 1:
                    print(f"  Using class weights: {dict(zip(unique, class_weights))}")
            else:
                sample_weights = None
                use_weights = False
            train_dates = valid_data.iloc[train_start_idx:train_end_idx].index
            test_dates = valid_data.iloc[test_start_idx:test_end_idx].index
            if self.config.USE_FEATURE_SELECTION and (not feature_selection_done):
                selected_features = self.feature_selector.select_features(
                    X_train_df, y_train, feature_cols, top_n=self.config.N_FEATURES_SELECT
                )
                feature_selection_done = True
            elif self.config.USE_FEATURE_SELECTION:
                selected_features = self.feature_selector.selected_features
            else:
                selected_features = feature_cols
            if selected_features is None or len(selected_features) == 0:
                print(f"Fold {fold_counter}: ERROR - No features selected, using all features")
                selected_features = feature_cols
            X_train_df = X_train_df[selected_features]
            X_test_df = X_test_df[selected_features]
            train_data_for_hmm = df.loc[train_dates]
            try:
                regime_detector.fit(train_data_for_hmm, regime_features)
                train_regimes = regime_detector.predict(train_data_for_hmm, regime_features)
                test_data_for_hmm = df.loc[test_dates]
                test_regimes = regime_detector.predict(test_data_for_hmm, regime_features)
                df.loc[train_dates, "regime"] = train_regimes
                df.loc[test_dates, "regime"] = test_regimes
                train_regimes_clean = np.nan_to_num(train_regimes, nan=1.0)
                test_regimes_clean = np.nan_to_num(test_regimes, nan=1.0)
                X_train_df["regime"] = train_regimes_clean
                X_test_df["regime"] = test_regimes_clean
            except Exception as e:
                print(f"  Warning: HMM failed for fold {fold_counter}: {str(e)}")
                pass
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train_df)
            X_test_scaled = scaler.transform(X_test_df)
            X_train = X_train_scaled.astype(np.float32)
            X_test = X_test_scaled.astype(np.float32)
            if fold_counter % 10 == 1:
                print(f"\nFold {fold_counter}")
                print("-" * 40)
                print(
                    f"Train: {train_dates[0].date()} to {train_dates[-1].date()} ({len(X_train)} days)"
                )
                print(
                    f"Test:  {test_dates[0].date()} to {test_dates[-1].date()} ({len(X_test)} days)"
                )
                print(f"Features: {X_train.shape[1]}")
            model = xgb.XGBClassifier(**self.config.XGB_PARAMS)
            if use_weights:
                model.fit(
                    X_train,
                    y_train,
                    sample_weight=sample_weights,
                    eval_set=[(X_train, y_train)],
                    verbose=False,
                )
            else:
                model.fit(X_train, y_train, eval_set=[(X_train, y_train)], verbose=False)
            y_pred = model.predict(X_test)
            y_prob = model.predict_proba(X_test)[:, 1]
            accuracy = accuracy_score(y_test, y_pred)
            if fold_counter % 10 == 1:
                current_feature_names = list(X_train_df.columns)
                feature_importance = pd.DataFrame(
                    {"feature": current_feature_names, "importance": model.feature_importances_}
                ).sort_values("importance", ascending=False)
                print(f"Accuracy: {accuracy:.4f}")
                print(
                    f"Top 5 features: {', '.join(feature_importance.head(5)['feature'].tolist())}"
                )
            for idx, pred, prob in zip(test_dates, y_pred, y_prob):
                all_predictions[idx] = pred
                all_probabilities[idx] = prob
            self.models.append(model)
            fold_results.append(
                {
                    "fold": fold_counter,
                    "train_start": train_dates[0],
                    "train_end": train_dates[-1],
                    "test_start": test_dates[0],
                    "test_end": test_dates[-1],
                    "train_size": len(X_train),
                    "test_size": len(X_test),
                    "accuracy": accuracy,
                    "mean_prob": y_prob.mean(),
                    "std_prob": y_prob.std(),
                }
            )
            train_end_idx += adjusted_refit
        results_df = pd.DataFrame(fold_results)
        print("\n" + "=" * 60)
        print("ROLLING WALK-FORWARD VALIDATION SUMMARY")
        print("=" * 60)
        print(f"Total refits: {len(results_df)}")
        print(f"Average accuracy: {results_df['accuracy'].mean():.4f}")
        print(f"Std accuracy: {results_df['accuracy'].std():.4f}")
        print(f"Min accuracy: {results_df['accuracy'].min():.4f}")
        print(f"Max accuracy: {results_df['accuracy'].max():.4f}")
        print("\nFirst 3 folds:")
        print(
            results_df[["fold", "test_start", "test_end", "accuracy"]]
            .head(3)
            .to_string(index=False)
        )
        print("\nLast 3 folds:")
        print(
            results_df[["fold", "test_start", "test_end", "accuracy"]]
            .tail(3)
            .to_string(index=False)
        )
        df["ml_prediction"] = all_predictions
        df["ml_probability"] = all_probabilities
        return (df, results_df)


class HybridBacktester:
    """
    Production-grade backtesting engine with hybrid P&L calculation.

    Key Features:
    1. Temporal Flow Control: Signal at T-1, execute at T
    2. Hybrid P&L: Different calculation for execution vs hold days
    3. Position Sizing: Avoids artificial leverage bug
    4. Transaction Costs: Realistic commission and slippage
    5. Benchmark Comparison: Tracks vs $URTH
    """

    def __init__(self, config):
        self.config = config
        self.trades = []
        self.daily_stats = []

    def run(self, df, df_benchmark):
        """
        Execute backtest with hybrid P&L calculation.

        P&L Logic:
        ----------
        Execution Day (position change):
          - P&L = (Open[T] / Open[T-1]) - 1
          - Uses Open prices because execution happens at Open

        Hold Day (no position change):
          - P&L = (Close[T] / Close[T-1]) - 1
          - Uses Close prices for mark-to-market

        This hybrid approach:
        - Captures overnight gap risk on execution
        - Avoids intraday noise on hold days
        - Matches real trading behavior
        """
        print("\n" + "=" * 60)
        print("RUNNING BACKTEST")
        print("=" * 60)
        start_date = pd.to_datetime(self.config.BACKTEST_START)
        end_date = pd.to_datetime(self.config.BACKTEST_END)
        df_bt = df[(df.index >= start_date) & (df.index <= end_date)].copy()
        df_bm = df_benchmark[
            (df_benchmark.index >= start_date) & (df_benchmark.index <= end_date)
        ].copy()
        if len(df_bt) == 0:
            raise ValueError(
                f"No data in backtest period {start_date} to {end_date}. Available data: {df.index.min()} to {df.index.max()}. Adjust BACKTEST_START to at least {df.index.min() + pd.Timedelta(days=300)}"
            )
        print(f"Backtest period: {df_bt.index[0].date()} to {df_bt.index[-1].date()}")
        print(f"Trading days: {len(df_bt)}")
        if self.config.USE_CVAR_STOP:
            print(f"\nCalculating rolling CVaR stop loss...")
            print(f"  Lookback: {self.config.CVAR_LOOKBACK} days")
            print(f"  Quantile: {self.config.CVAR_QUANTILE}")
            print(f"  Multiplier: {self.config.CVAR_MULTIPLIER}x")
            df_bt["cvar_threshold"] = np.nan
            df_bt["stop_threshold"] = np.nan
            print(f"✓ CVaR will be calculated on strategy returns during backtest")
        else:
            df_bt["stop_threshold"] = np.nan
        capital = self.config.INITIAL_CAPITAL
        cash = capital
        position = 0
        position_value = 0
        prev_signal = 0
        entry_price = None
        entry_date = None
        portfolio_values = []
        positions_held = []
        cash_reserves = []
        df_bm["bm_return"] = df_bm["close"].pct_change()
        benchmark_values = [capital]
        for i in range(len(df_bt)):
            date = df_bt.index[i]
            row = df_bt.iloc[i]
            signal = row["ml_prediction"] if pd.notna(row["ml_prediction"]) else 0
            signal = int(signal)
            target_direction = 1 if signal == 1 else -1
            stop_triggered = False
            stop_threshold = np.nan
            if (
                self.config.USE_CVAR_STOP
                and position != 0
                and (entry_price is not None)
                and (i >= self.config.CVAR_LOOKBACK)
            ):
                lookback_returns = pd.Series(
                    [
                        self.daily_stats[j]["portfolio_return"]
                        for j in range(
                            max(0, len(self.daily_stats) - self.config.CVAR_LOOKBACK),
                            len(self.daily_stats),
                        )
                    ]
                )
                if len(lookback_returns) >= 20:
                    var_threshold = lookback_returns.quantile(1 - self.config.CVAR_QUANTILE)
                    tail_returns = lookback_returns[lookback_returns <= var_threshold]
                    if len(tail_returns) > 0:
                        cvar = tail_returns.mean()
                        stop_threshold = cvar * self.config.CVAR_MULTIPLIER
                current_price = row["close"]
                position_return = (current_price / entry_price - 1) * np.sign(position)
                if pd.notna(stop_threshold) and position_return < stop_threshold:
                    target_direction = 0
                    stop_triggered = True
                    print(
                        f"CVaR Stop: {date.date()} P&L={position_return:.2%} < {stop_threshold:.2%}"
                    )
            execute_trade = i == 0 or target_direction != prev_signal
            if execute_trade:
                old_position = position
                old_position_value = position_value
                if i > 0:
                    size_price = df_bt.iloc[i - 1]["close"]
                else:
                    size_price = row["close"]
                if self.config.COMPOUNDING_MODE:
                    portfolio_value = cash + old_position_value
                    target_notional = portfolio_value * self.config.LEVERAGE * target_direction
                else:
                    target_notional = capital * self.config.LEVERAGE * target_direction
                target_shares = target_notional / size_price
                trade_shares = target_shares - old_position
                if trade_shares != 0:
                    execution_price = row["open"]
                    trade_value = abs(trade_shares * execution_price)
                    commission = trade_value * self.config.COMMISSION_PCT
                    slippage = trade_value * (self.config.SLIPPAGE_BPS / 10000)
                    total_costs = commission + slippage
                    if trade_shares > 0:
                        cash -= trade_shares * execution_price + total_costs
                    else:
                        cash += abs(trade_shares * execution_price) - total_costs
                    position = target_shares
                    if position != 0:
                        entry_price = execution_price
                        entry_date = date
                    else:
                        entry_price = None
                        entry_date = None
                    self.trades.append(
                        {
                            "date": date,
                            "signal": signal,
                            "direction": target_direction,
                            "shares": trade_shares,
                            "price": execution_price,
                            "value": trade_value,
                            "commission": commission,
                            "slippage": slippage,
                            "total_cost": total_costs,
                            "stop_triggered": stop_triggered,
                        }
                    )
            if i > 0:
                prev_close = df_bt.iloc[i - 1]["close"]
                if execute_trade:
                    gap_return = row["open"] / prev_close - 1
                    gap_pnl = old_position_value * gap_return
                    intraday_return = row["close"] / row["open"] - 1
                    intraday_pnl = position * row["open"] * intraday_return
                    position_pnl = gap_pnl + intraday_pnl
                    position_value = position * row["close"] if position != 0 else 0
                    if portfolio_values:
                        portfolio_return = position_pnl / portfolio_values[-1]
                    else:
                        portfolio_return = 0
                else:
                    pnl_return = row["close"] / prev_close - 1
                    position_pnl = position_value * pnl_return
                    position_value = position * row["close"] if position != 0 else 0
                    if portfolio_values:
                        portfolio_return = position_pnl / portfolio_values[-1]
                    else:
                        portfolio_return = 0
            else:
                portfolio_return = 0
                position_pnl = 0
                position_value = position * row["close"] if position != 0 else 0
            portfolio_value = cash + position_value
            portfolio_values.append(portfolio_value)
            positions_held.append(position)
            cash_reserves.append(cash)
            if date in df_bm.index:
                bm_ret = df_bm.loc[date, "bm_return"]
                if pd.notna(bm_ret):
                    benchmark_values.append(benchmark_values[-1] * (1 + bm_ret))
                else:
                    benchmark_values.append(benchmark_values[-1])
            else:
                benchmark_values.append(benchmark_values[-1])
            self.daily_stats.append(
                {
                    "date": date,
                    "portfolio_value": portfolio_value,
                    "cash": cash,
                    "position_value": position_value,
                    "position_shares": position,
                    "signal": signal,
                    "execute_trade": execute_trade,
                    "portfolio_return": portfolio_return,
                    "benchmark_value": benchmark_values[-1],
                }
            )
            prev_signal = target_direction
        results_df = pd.DataFrame(self.daily_stats)
        results_df = results_df.set_index("date")
        results_df["benchmark_return"] = results_df["benchmark_value"].pct_change()
        results_df["cumulative_return"] = results_df["portfolio_value"] / capital - 1
        results_df["cumulative_benchmark"] = results_df["benchmark_value"] / capital - 1
        print(f"\n✓ Backtest complete")
        print(f"Total trades: {len(self.trades)}")
        print(f"Final portfolio value: ${results_df['portfolio_value'].iloc[-1]:,.2f}")
        print(f"Total return: {results_df['cumulative_return'].iloc[-1] * 100:.2f}%")
        if self.trades:
            total_commissions = sum([t["commission"] for t in self.trades])
            total_slippage = sum([t["slippage"] for t in self.trades])
            total_costs = total_commissions + total_slippage
            cost_drag = total_costs / capital * 100
            avg_cost_per_trade = total_costs / len(self.trades)
            print(f"\nTransaction Cost Analysis:")
            print(f"  Total commissions:  ${total_commissions:,.2f}")
            print(f"  Total slippage:     ${total_slippage:,.2f}")
            print(f"  Total costs:        ${total_costs:,.2f}")
            print(f"  Cost drag:          {cost_drag:.2f}% of initial capital")
            print(f"  Avg cost/trade:     ${avg_cost_per_trade:.2f}")
        if self.config.USE_CVAR_STOP and self.trades:
            stops_triggered = [t for t in self.trades if t["stop_triggered"]]
            stop_rate = len(stops_triggered) / len(self.trades) * 100
            print(f"\nStop Loss Effectiveness:")
            print(
                f"  Stops triggered:    {len(stops_triggered)}/{len(self.trades)} ({stop_rate:.1f}%)"
            )
            if stops_triggered:
                stop_dates = [t["date"] for t in stops_triggered]
                print(f"  First stop:         {stop_dates[0].date()}")
                print(f"  Last stop:          {stop_dates[-1].date()}")
        return results_df

    def get_trades_df(self):
        """Return trades as DataFrame."""
        if not self.trades:
            return pd.DataFrame()
        return pd.DataFrame(self.trades).set_index("date")


class RobustnessMetrics:
    """
    Comprehensive statistical validation metrics.

    These metrics help detect overfitting and validate strategy robustness:
    1. Deflated Sharpe Ratio (DSR): Adjusts for multiple testing
    2. Probabilistic Sharpe Ratio (PSR): Statistical significance of SR
    3. Walk-Forward Efficiency (WFE): In-sample vs out-of-sample
    4. Hurst Exponent: Detect mean reversion vs trending
    5. Conditional Drawdown at Risk (CDaR): Tail drawdown risk
    6. Global Robustness Index (GRI): Combined score
    """

    def __init__(self, config):
        self.config = config

    def deflated_sharpe_ratio(self, returns, n_tests=None):
        """
        Deflated Sharpe Ratio (Bailey et al. 2014).

        Adjusts Sharpe ratio for multiple testing bias.
        Accounts for the fact that we tried N different strategies.

        Formula:
        DSR = SR / sqrt(1 + (N-1) * (γ + κ/4))

        where:
        - SR: Observed Sharpe ratio
        - N: Number of tests/strategies tried
        - γ: Skewness adjustment
        - κ: Kurtosis adjustment
        """
        if n_tests is None:
            n_tests = self.config.N_TESTS_ADJUSTMENT
        returns_clean = returns.dropna()
        if len(returns_clean) < 30:
            return np.nan
        sr = returns_clean.mean() / returns_clean.std() * np.sqrt(252)
        skew = returns_clean.skew()
        kurt = returns_clean.kurtosis()
        adjustment = np.sqrt(1 + (n_tests - 1) * (skew**2 + kurt / 4))
        dsr = sr / adjustment if adjustment > 0 else np.nan
        return dsr

    def probabilistic_sharpe_ratio(self, returns, benchmark_sr=0):
        """
        Probabilistic Sharpe Ratio (PSR).

        Probability that true Sharpe ratio exceeds benchmark.
        Uses t-distribution to account for estimation error.

        PSR > 0.95 indicates statistical significance at 95% confidence.
        """
        returns_clean = returns.dropna()
        if len(returns_clean) < 30:
            return np.nan
        sr_obs = returns_clean.mean() / returns_clean.std() * np.sqrt(252)
        n = len(returns_clean)
        skew = returns_clean.skew()
        kurt = returns_clean.kurtosis()
        sr_std = np.sqrt((1 + 0.5 * sr_obs**2 - skew * sr_obs + (kurt - 3) / 4 * sr_obs**2) / n)
        test_stat = (sr_obs - benchmark_sr) / sr_std
        psr = t.cdf(test_stat, df=n - 1)
        return psr

    def walk_forward_efficiency(self, is_sharpe, oos_sharpe):
        """
        Walk-Forward Efficiency (WFE).

        Ratio of out-of-sample Sharpe to in-sample Sharpe.

        WFE > 0.5: Decent generalization
        WFE < 0.5: Likely overfitting
        WFE > 1.0: Strategy improves out-of-sample (rare but good)
        """
        if is_sharpe == 0 or np.isnan(is_sharpe) or np.isnan(oos_sharpe):
            return np.nan
        wfe = oos_sharpe / is_sharpe
        return wfe

    def hurst_exponent(self, returns):
        """
        Hurst Exponent for entire return series.

        H < 0.5: Mean-reverting
        H = 0.5: Random walk
        H > 0.5: Trending/persistent
        """
        returns_clean = returns.dropna()
        if len(returns_clean) < 100:
            return np.nan
        lags = range(2, min(100, len(returns_clean) // 2))
        rs_values = []
        for lag in lags:
            n_blocks = len(returns_clean) // lag
            if n_blocks < 2:
                continue
            rs_lag = []
            for i in range(n_blocks):
                block = returns_clean.iloc[i * lag : (i + 1) * lag]
                mean_adj = block - block.mean()
                cum_sum = mean_adj.cumsum()
                R = cum_sum.max() - cum_sum.min()
                S = block.std()
                if S > 0:
                    rs_lag.append(R / S)
            if rs_lag:
                rs_values.append((lag, np.mean(rs_lag)))
        if len(rs_values) < 5:
            return np.nan
        lags_arr = np.array([x[0] for x in rs_values])
        rs_arr = np.array([x[1] for x in rs_values])
        log_lags = np.log(lags_arr)
        log_rs = np.log(rs_arr)
        hurst = np.polyfit(log_lags, log_rs, 1)[0]
        return hurst

    def conditional_drawdown_at_risk(self, returns, quantile=None):
        """
        Conditional Drawdown at Risk (CDaR).

        Average of worst drawdowns beyond VaR threshold.
        Measures tail risk of drawdowns.
        """
        if quantile is None:
            quantile = self.config.CDAR_QUANTILE
        returns_clean = returns.dropna()
        if len(returns_clean) < 30:
            return np.nan
        cum_returns = (1 + returns_clean).cumprod()
        running_max = cum_returns.expanding().max()
        drawdown = (cum_returns - running_max) / running_max
        threshold = drawdown.quantile(1 - quantile)
        tail_drawdowns = drawdown[drawdown <= threshold]
        if len(tail_drawdowns) == 0:
            return np.nan
        cdar = tail_drawdowns.mean()
        return abs(cdar)

    def rolling_information_coefficient(self, predictions, returns, window=60):
        """
        Rolling Information Coefficient (IC).

        Measures correlation between predictions and realized returns.
        Rolling window ensures we only use past data at each point.

        High IC = consistent predictive power
        IC near 0 = no predictive ability
        Negative IC = inverse predictions (wrong direction)

        Parameters:
        -----------
        predictions : pd.Series
            Model predictions (probabilities or signals)
        returns : pd.Series
            Realized returns (already properly shifted)
        window : int
            Rolling window size

        Returns:
        --------
        pd.Series : Rolling IC values
        float : Mean IC
        """
        df_ic = pd.DataFrame({"pred": predictions, "ret": returns}).dropna()
        if len(df_ic) < window:
            return (pd.Series(dtype=float), np.nan)
        rolling_ic = df_ic["pred"].rolling(window).corr(df_ic["ret"])
        mean_ic = rolling_ic.dropna().mean()
        return (rolling_ic, mean_ic)

    def global_robustness_index(self, returns, is_sharpe=None, oos_sharpe=None):
        """
        Global Robustness Index (GRI).

        Combines multiple metrics into single score [0, 1].

        Components:
        1. DSR (normalized)
        2. PSR
        3. WFE (normalized)
        4. Hurst proximity to 0.5 (normalized)
        5. CDaR (inverted and normalized)

        GRI > 0.65: Strategy is robust
        GRI < 0.5: Strategy may be overfit
        """
        dsr = self.deflated_sharpe_ratio(returns)
        psr = self.probabilistic_sharpe_ratio(returns)
        hurst = self.hurst_exponent(returns)
        cdar = self.conditional_drawdown_at_risk(returns)
        if is_sharpe and oos_sharpe:
            wfe = self.walk_forward_efficiency(is_sharpe, oos_sharpe)
        else:
            wfe = np.nan
        scores = []
        if not np.isnan(dsr):
            dsr_norm = min(max(dsr / 2, 0), 1)
            scores.append(dsr_norm)
        if not np.isnan(psr):
            scores.append(psr)
        if not np.isnan(wfe):
            wfe_norm = min(max(wfe, 0), 1)
            scores.append(wfe_norm)
        if not np.isnan(hurst):
            hurst_norm = 1 - 2 * abs(hurst - 0.5)
            hurst_norm = max(hurst_norm, 0)
            scores.append(hurst_norm)
        if not np.isnan(cdar):
            cdar_norm = 1 - min(abs(cdar) / 0.5, 1)
            scores.append(cdar_norm)
        if len(scores) == 0:
            return np.nan
        gri = np.mean(scores)
        return gri

    def compute_all_metrics(
        self, returns, predictions=None, realized_returns=None, is_sharpe=None, oos_sharpe=None
    ):
        """
        Compute all robustness metrics.

        Parameters:
        -----------
        returns : pd.Series
            Portfolio returns
        predictions : pd.Series, optional
            Model predictions (for IC calculation)
        realized_returns : pd.Series, optional
            Realized returns aligned with predictions (for IC)
        is_sharpe : float, optional
            In-sample Sharpe ratio
        oos_sharpe : float, optional
            Out-of-sample Sharpe ratio
        """
        print("\n" + "=" * 60)
        print("ROBUSTNESS METRICS")
        print("=" * 60)
        metrics = {}
        returns_clean = returns.dropna()
        metrics["sharpe_ratio"] = returns_clean.mean() / returns_clean.std() * np.sqrt(252)
        metrics["annual_return"] = returns_clean.mean() * 252
        metrics["annual_volatility"] = returns_clean.std() * np.sqrt(252)
        cum_returns = (1 + returns_clean).cumprod()
        running_max = cum_returns.expanding().max()
        drawdown = (cum_returns - running_max) / running_max
        metrics["max_drawdown"] = drawdown.min()
        metrics["deflated_sharpe"] = self.deflated_sharpe_ratio(returns)
        metrics["probabilistic_sharpe"] = self.probabilistic_sharpe_ratio(returns)
        metrics["hurst_exponent"] = self.hurst_exponent(returns)
        metrics["cdar"] = self.conditional_drawdown_at_risk(returns)
        if is_sharpe and oos_sharpe:
            metrics["walk_forward_efficiency"] = self.walk_forward_efficiency(is_sharpe, oos_sharpe)
        else:
            metrics["walk_forward_efficiency"] = np.nan
        if predictions is not None and realized_returns is not None:
            rolling_ic, mean_ic = self.rolling_information_coefficient(
                predictions, realized_returns
            )
            metrics["mean_ic"] = mean_ic
            metrics["ic_std"] = rolling_ic.dropna().std()
            metrics["ic_sharpe"] = mean_ic / metrics["ic_std"] if metrics["ic_std"] > 0 else np.nan
        else:
            metrics["mean_ic"] = np.nan
            metrics["ic_std"] = np.nan
            metrics["ic_sharpe"] = np.nan
        metrics["global_robustness_index"] = self.global_robustness_index(
            returns, is_sharpe, oos_sharpe
        )
        print(f"\nBasic Statistics:")
        print(f"  Sharpe Ratio:       {metrics['sharpe_ratio']:.4f}")
        print(f"  Annual Return:      {metrics['annual_return'] * 100:.2f}%")
        print(f"  Annual Volatility:  {metrics['annual_volatility'] * 100:.2f}%")
        print(f"  Max Drawdown:       {metrics['max_drawdown'] * 100:.2f}%")
        print(f"\nRobustness Metrics:")
        print(f"  Deflated Sharpe:    {metrics['deflated_sharpe']:.4f}")
        print(f"  Probabilistic SR:   {metrics['probabilistic_sharpe']:.4f}")
        print(f"  WF Efficiency:      {metrics['walk_forward_efficiency']:.4f}")
        print(f"  Hurst Exponent:     {metrics['hurst_exponent']:.4f}")
        print(f"  CDaR (95%):         {metrics['cdar'] * 100:.2f}%")
        print(f"\nPredictive Power:")
        print(f"  Mean IC:            {metrics['mean_ic']:.4f}")
        print(f"  IC Std Dev:         {metrics['ic_std']:.4f}")
        print(f"  IC Sharpe:          {metrics['ic_sharpe']:.4f}")
        print(f"\n{'=' * 60}")
        print(f"GLOBAL ROBUSTNESS INDEX: {metrics['global_robustness_index']:.4f}")
        print(f"{'=' * 60}")
        if metrics["global_robustness_index"] > 0.65:
            print("✓ Strategy passes robustness threshold (GRI > 0.65)")
        else:
            print("✗ Strategy fails robustness threshold (GRI < 0.65)")
            print("  → May indicate overfitting or insufficient validation")
        return metrics


class MultiTickerValidator:
    """
    Cross-validation framework for testing strategy robustness across assets.

    Tests strategy on multiple tickers to ensure:
    1. Not overfit to single asset
    2. Generalizes across different volatility regimes
    3. Robust to different market microstructures

    Success criteria: GRI > 0.60 on 70%+ of test tickers
    """

    def __init__(self, base_config):
        """
        Initialize with base configuration template.

        Parameters:
        -----------
        base_config : StrategyConfig
            Base configuration to clone for each ticker
        """
        self.base_config = base_config
        self.results = {}
        self.metrics_summary = None

    def validate_ticker(self, ticker, config_overrides=None):
        """
        Run full backtest on single ticker.

        Parameters:
        -----------
        ticker : str
            Ticker symbol to test
        config_overrides : dict, optional
            Custom config parameters for this ticker

        Returns:
        --------
        dict : Results including GRI, Sharpe, drawdown, etc.
        """
        print(f"\n{'=' * 70}")
        print(f"VALIDATING TICKER: {ticker}")
        print("=" * 70)
        config = StrategyConfig()
        config.SYMBOL_MAIN = ticker
        if config_overrides:
            for key, value in config_overrides.items():
                setattr(config, key, value)
        try:
            fetcher = DataFetcher()
            feature_engine = FeatureEngine(config)
            ml_model = MLTradingModel(config)
            backtester = HybridBacktester(config)
            metrics_calc = RobustnessMetrics(config)
            print(f"\nFetching data for {ticker}...")
            df_main = fetcher.fetch_ohlcv(ticker, config.START_DATE, config.END_DATE)
            if len(df_main) < 300:
                return {"status": "failed", "error": f"Insufficient data: {len(df_main)} days"}
            corr_assets = {}
            for symbol in config.SYMBOLS_CORR:
                df_corr = fetcher.fetch_ohlcv(symbol, config.START_DATE, config.END_DATE)
                if not df_corr.empty:
                    corr_assets[symbol.lower()] = df_corr
            df_benchmark = fetcher.fetch_ohlcv(config.BENCHMARK, config.START_DATE, config.END_DATE)
            df_features = feature_engine.compute_all_features(df_main, corr_assets)
            df_ml, wf_results = ml_model.walk_forward_validation(df_features)
            results_df = backtester.run(df_ml, df_benchmark)
            mid_point = len(results_df) // 2
            is_returns = results_df["portfolio_return"].iloc[:mid_point]
            oos_returns = results_df["portfolio_return"].iloc[mid_point:]
            is_sharpe = (
                is_returns.mean() / is_returns.std() * np.sqrt(252)
                if is_returns.std() > 0
                else np.nan
            )
            oos_sharpe = (
                oos_returns.mean() / oos_returns.std() * np.sqrt(252)
                if oos_returns.std() > 0
                else np.nan
            )
            robustness = metrics_calc.compute_all_metrics(
                results_df["portfolio_return"], is_sharpe=is_sharpe, oos_sharpe=oos_sharpe
            )
            result = {
                "status": "success",
                "ticker": ticker,
                "total_return": results_df["cumulative_return"].iloc[-1],
                "sharpe_ratio": robustness["sharpe_ratio"],
                "max_drawdown": robustness["max_drawdown"],
                "deflated_sharpe": robustness["deflated_sharpe"],
                "probabilistic_sharpe": robustness["probabilistic_sharpe"],
                "walk_forward_efficiency": robustness["walk_forward_efficiency"],
                "hurst_exponent": robustness["hurst_exponent"],
                "cdar": robustness["cdar"],
                "global_robustness_index": robustness["global_robustness_index"],
                "total_trades": len(backtester.trades),
                "trading_days": len(results_df),
                "mean_ic": robustness.get("mean_ic", np.nan),
                "config": {
                    "refit_freq": config.REFIT_FREQUENCY,
                    "prediction_horizon": config.PREDICTION_HORIZON,
                    "leverage": config.LEVERAGE,
                },
            }
            print(f"\n✓ {ticker} validation complete")
            print(f"  GRI: {result['global_robustness_index']:.4f}")
            print(f"  Sharpe: {result['sharpe_ratio']:.4f}")
            print(f"  Return: {result['total_return'] * 100:.2f}%")
            return result
        except Exception as e:
            print(f"\n✗ {ticker} validation failed: {str(e)}")
            import traceback

            traceback.print_exc()
            return {"status": "failed", "ticker": ticker, "error": str(e)}

    def run_validation(self, tickers, config_overrides=None, gri_threshold=0.6):
        """
        Run validation across multiple tickers.

        Parameters:
        -----------
        tickers : list
            List of ticker symbols to test
        config_overrides : dict, optional
            Config overrides to apply to all tickers
        gri_threshold : float
            Minimum GRI for success (default 0.60)

        Returns:
        --------
        pd.DataFrame : Summary of results across all tickers
        """
        print("\n" + "=" * 70)
        print("MULTI-TICKER VALIDATION")
        print("=" * 70)
        print(f"Tickers to test: {', '.join(tickers)}")
        print(f"GRI threshold: {gri_threshold}")
        print(f"Success criteria: 70%+ pass threshold")
        for ticker in tickers:
            result = self.validate_ticker(ticker, config_overrides)
            self.results[ticker] = result
        summary_data = []
        for ticker, result in self.results.items():
            if result["status"] == "success":
                summary_data.append(
                    {
                        "Ticker": ticker,
                        "Total Return (%)": result["total_return"] * 100,
                        "Sharpe Ratio": result["sharpe_ratio"],
                        "Max DD (%)": result["max_drawdown"] * 100,
                        "DSR": result["deflated_sharpe"],
                        "PSR": result["probabilistic_sharpe"],
                        "WFE": result["walk_forward_efficiency"],
                        "Hurst": result["hurst_exponent"],
                        "CDaR (%)": result["cdar"] * 100,
                        "GRI": result["global_robustness_index"],
                        "Pass": "✓" if result["global_robustness_index"] > gri_threshold else "✗",
                        "Trades": result["total_trades"],
                    }
                )
            else:
                summary_data.append(
                    {
                        "Ticker": ticker,
                        "Total Return (%)": np.nan,
                        "Sharpe Ratio": np.nan,
                        "Max DD (%)": np.nan,
                        "DSR": np.nan,
                        "PSR": np.nan,
                        "WFE": np.nan,
                        "Hurst": np.nan,
                        "CDaR (%)": np.nan,
                        "GRI": np.nan,
                        "Pass": "✗",
                        "Trades": 0,
                    }
                )
        self.metrics_summary = pd.DataFrame(summary_data)
        successful = self.metrics_summary[self.metrics_summary["GRI"] > gri_threshold]
        success_rate = len(successful) / len(tickers)
        print("\n" + "=" * 70)
        print("VALIDATION SUMMARY")
        print("=" * 70)
        print(self.metrics_summary.to_string(index=False))
        print(f"\n{'=' * 70}")
        print(f"SUCCESS RATE: {success_rate * 100:.1f}% ({len(successful)}/{len(tickers)} tickers)")
        print("=" * 70)
        if success_rate >= 0.7:
            print("✓ STRATEGY IS ROBUST - Generalizes across multiple assets")
        elif success_rate >= 0.5:
            print("⚠ STRATEGY IS MARGINAL - Consider reducing complexity")
        else:
            print("✗ STRATEGY OVERFITS - Fails to generalize")
        if len(successful) > 0:
            print(f"\nSuccessful tickers (GRI > {gri_threshold}):")
            print(f"  Mean GRI: {successful['GRI'].mean():.4f}")
            print(f"  Mean Sharpe: {successful['Sharpe Ratio'].mean():.4f}")
            print(f"  Mean Return: {successful['Total Return (%)'].mean():.2f}%")
            print(f"  Mean Max DD: {successful['Max DD (%)'].mean():.2f}%")
        failed = self.metrics_summary[
            self.metrics_summary["GRI"].notna() & (self.metrics_summary["GRI"] <= gri_threshold)
        ]
        if len(failed) > 0:
            print(f"\nFailed tickers (GRI ≤ {gri_threshold}):")
            for _, row in failed.iterrows():
                print(f"  {row['Ticker']}: GRI={row['GRI']:.4f}, Sharpe={row['Sharpe Ratio']:.4f}")
        return self.metrics_summary

    def save_results(self, output_dir="results/machine-learning-mstr-market-regimes/"):
        """Save validation results to CSV."""
        if self.metrics_summary is None:
            print("No results to save. Run run_validation() first.")
            return
        summary_path = f"{output_dir}/multi_ticker_validation.csv"
        self.metrics_summary.to_csv(summary_path, index=False)
        print(f"✓ Validation summary saved: {summary_path}")
        import json

        detailed_path = f"{output_dir}/multi_ticker_detailed.json"
        with open(detailed_path, "w") as f:
            json.dump(self.results, f, indent=2, default=str)
        print(f"✓ Detailed results saved: {detailed_path}")

    def get_best_ticker(self):
        """Return ticker with highest GRI."""
        if self.metrics_summary is None:
            return None
        best = self.metrics_summary.loc[self.metrics_summary["GRI"].idxmax()]
        return best["Ticker"]


def create_performance_report(results_df, benchmark_df, config):
    """Create comprehensive performance visualization."""
    fig, axes = plt.subplots(3, 2, figsize=config.PLOT_FIGSIZE, dpi=config.PLOT_DPI)
    fig.suptitle("ML Regime-Adaptive Strategy Performance", fontsize=16, fontweight="bold")
    ax = axes[0, 0]
    ax.plot(results_df.index, results_df["cumulative_return"] * 100, label="Strategy", linewidth=2)
    ax.plot(
        results_df.index,
        results_df["cumulative_benchmark"] * 100,
        label=f"Benchmark ({config.BENCHMARK})",
        linewidth=2,
        alpha=0.7,
    )
    ax.set_title("Cumulative Returns")
    ax.set_ylabel("Return (%)")
    ax.legend()
    ax.grid(alpha=0.3)
    ax = axes[0, 1]
    cum_ret = (1 + results_df["portfolio_return"]).cumprod()
    running_max = cum_ret.expanding().max()
    drawdown = (cum_ret - running_max) / running_max * 100
    ax.fill_between(drawdown.index, 0, drawdown, alpha=0.3, color="red")
    ax.plot(drawdown.index, drawdown, color="red", linewidth=1)
    ax.set_title("Strategy Drawdown")
    ax.set_ylabel("Drawdown (%)")
    ax.grid(alpha=0.3)
    ax = axes[1, 0]
    rolling_sharpe = (
        results_df["portfolio_return"].rolling(60).mean()
        / results_df["portfolio_return"].rolling(60).std()
        * np.sqrt(252)
    )
    ax.plot(rolling_sharpe.index, rolling_sharpe, linewidth=1.5)
    ax.axhline(y=0, color="black", linestyle="--", alpha=0.5)
    ax.set_title("Rolling 60-Day Sharpe Ratio")
    ax.set_ylabel("Sharpe Ratio")
    ax.grid(alpha=0.3)
    ax = axes[1, 1]
    monthly_returns = (
        results_df["portfolio_return"].resample("ME").apply(lambda x: (1 + x).prod() - 1) * 100
    )
    monthly_pivot = monthly_returns.to_frame("return")
    monthly_pivot["year"] = monthly_pivot.index.year
    monthly_pivot["month"] = monthly_pivot.index.month
    pivot = monthly_pivot.pivot(index="year", columns="month", values="return")
    im = ax.imshow(pivot.values, cmap="RdYlGn", aspect="auto", vmin=-10, vmax=10)
    ax.set_xticks(range(12))
    ax.set_xticklabels(["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
    ax.set_yticks(range(len(pivot)))
    ax.set_yticklabels(pivot.index)
    ax.set_title("Monthly Returns Heatmap (%)")
    plt.colorbar(im, ax=ax)
    ax = axes[2, 0]
    position_counts = results_df["signal"].value_counts()
    ax.bar(position_counts.index, position_counts.values, color=["red", "green"])
    ax.set_title("Position Distribution")
    ax.set_xlabel("Signal (0=Short, 1=Long)")
    ax.set_ylabel("Days")
    ax.grid(alpha=0.3)
    ax = axes[2, 1]
    ax.hist(results_df["portfolio_return"] * 100, bins=50, alpha=0.7, edgecolor="black")
    ax.axvline(
        x=results_df["portfolio_return"].mean() * 100,
        color="red",
        linestyle="--",
        linewidth=2,
        label="Mean",
    )
    ax.set_title("Daily Return Distribution")
    ax.set_xlabel("Return (%)")
    ax.set_ylabel("Frequency")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    return fig


def main():
    """
    Main execution flow with all improvements integrated.
    """
    print("=" * 60)
    print("ML REGIME-ADAPTIVE TRADING STRATEGY")
    print("WITH FEATURE SELECTION AND OVERFITTING PREVENTION")
    print("=" * 60)
    print(f"\nConfiguration:")
    print(f"  Asset: {StrategyConfig.SYMBOL_MAIN}")
    print(f"  Period: {StrategyConfig.BACKTEST_START} to {StrategyConfig.BACKTEST_END}")
    print(f"  Initial Capital: ${StrategyConfig.INITIAL_CAPITAL:,.0f}")
    print(f"  Feature Selection: {StrategyConfig.USE_FEATURE_SELECTION}")
    print(f"  Features to Select: {StrategyConfig.N_FEATURES_SELECT}")
    config = StrategyConfig()
    fetcher = DataFetcher()
    feature_engine = FeatureEngine(config)
    ml_model = MLTradingModel(config)
    backtester = HybridBacktester(config)
    metrics_calc = RobustnessMetrics(config)
    print("\n" + "=" * 60)
    print("DATA ACQUISITION")
    print("=" * 60)
    df_main = fetcher.fetch_ohlcv(config.SYMBOL_MAIN, config.START_DATE, config.END_DATE)
    corr_assets = {}
    for symbol in config.SYMBOLS_CORR:
        df_corr = fetcher.fetch_ohlcv(symbol, config.START_DATE, config.END_DATE)
        if not df_corr.empty:
            corr_assets[symbol.lower()] = df_corr
    df_benchmark = fetcher.fetch_ohlcv(config.BENCHMARK, config.START_DATE, config.END_DATE)
    df_features = feature_engine.compute_all_features(df_main, corr_assets)
    df_ml, wf_results = ml_model.walk_forward_validation(df_features)
    results_df = backtester.run(df_ml, df_benchmark)
    trades_df = backtester.get_trades_df()
    mid_point = len(results_df) // 2
    is_returns = results_df["portfolio_return"].iloc[:mid_point]
    oos_returns = results_df["portfolio_return"].iloc[mid_point:]
    is_sharpe = (
        is_returns.mean() / is_returns.std() * np.sqrt(252) if is_returns.std() > 0 else np.nan
    )
    oos_sharpe = (
        oos_returns.mean() / oos_returns.std() * np.sqrt(252) if oos_returns.std() > 0 else np.nan
    )
    common_dates = results_df.index.intersection(df_ml.index)
    if len(common_dates) > 60:
        predictions_for_ic = df_ml.loc[common_dates, "ml_probability"]
        realized_returns_for_ic = df_ml.loc[common_dates, "target_return"]
    else:
        predictions_for_ic = None
        realized_returns_for_ic = None
    robustness_metrics = metrics_calc.compute_all_metrics(
        results_df["portfolio_return"],
        predictions=predictions_for_ic,
        realized_returns=realized_returns_for_ic,
        is_sharpe=is_sharpe,
        oos_sharpe=oos_sharpe,
    )
    print("\n" + "=" * 60)
    print("GENERATING PERFORMANCE REPORT")
    print("=" * 60)
    fig = create_performance_report(results_df, df_benchmark, config)
    output_dir = "results/machine-learning-mstr-market-regimes/"
    fig_path = f"{output_dir}/ml_regime_strategy_performance.png"
    fig.savefig(fig_path, dpi=config.PLOT_DPI, bbox_inches="tight")
    print(f"✓ Performance chart saved: {fig_path}")
    results_csv = f"{output_dir}/backtest_results.csv"
    results_df.to_csv(results_csv)
    print(f"✓ Backtest results saved: {results_csv}")
    if not trades_df.empty:
        trades_csv = f"{output_dir}/trades.csv"
        trades_df.to_csv(trades_csv)
        print(f"✓ Trades saved: {trades_csv}")
    wf_csv = f"{output_dir}/walk_forward_results.csv"
    wf_results.to_csv(wf_csv, index=False)
    print(f"✓ Walk-forward results saved: {wf_csv}")
    metrics_df = pd.DataFrame([robustness_metrics])
    metrics_csv = f"{output_dir}/robustness_metrics.csv"
    metrics_df.to_csv(metrics_csv, index=False)
    print(f"✓ Robustness metrics saved: {metrics_csv}")
    print("\n" + "=" * 60)
    print("EXECUTION COMPLETE")
    print("=" * 60)
    print(f"\nFinal Results:")
    print(f"  Total Return:    {results_df['cumulative_return'].iloc[-1] * 100:.2f}%")
    print(f"  Sharpe Ratio:    {robustness_metrics['sharpe_ratio']:.4f}")
    print(f"  Max Drawdown:    {robustness_metrics['max_drawdown'] * 100:.2f}%")
    print(f"  Total Trades:    {len(trades_df)}")
    print(f"  GRI Score:       {robustness_metrics['global_robustness_index']:.4f}")
    return (results_df, trades_df, robustness_metrics)


def run_multi_ticker_validation():
    """
    Run multi-ticker validation to test generalization.
    Call this AFTER main() to test on different assets.
    """
    test_tickers = [
        "MSTR",
        "COIN",
        "NVDA",
        "TSLA",
        "AMZN",
        "AMD",
        "NFLX",
        "INTC",
        "QCOM",
        "MU",
        "BIDU",
        "JD",
        "PDD",
        "BABA",
        "RACE",
        "MARA",
        "RIOT",
        "SOFI",
        "UPST",
        "IWM",
        "QQQ",
        "DIA",
        "EEM",
        "XLK",
        "XLF",
        "XLY",
        "XLI",
        "XLE",
        "XLB",
        "XLV",
        "XLU",
    ]
    validator = MultiTickerValidator(StrategyConfig())
    summary = validator.run_validation(tickers=test_tickers, gri_threshold=0.6)
    validator.save_results()
    best = validator.get_best_ticker()
    print(f"\nBest performing ticker: {best}")
    return summary


if __name__ == "__main__":
    results, trades, metrics = main()
    multi_ticker_summary = run_multi_ticker_validation()
